# Stacked BiLSTM

ByteLevel BPE → Embedding 128 → 4-layer BiLSTM, hidden 256 per direction → Dropout → Linear 512 → 3.

## Setup



In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



/content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"



In [3]:
import gc
import json
import platform
import uuid
from importlib.metadata import version

import pandas as pd
import torch
from tokenizers import Tokenizer

from src.data_io import (get_data_dir, load_manifest, load_data_splits,
                         save_json, save_predictions, file_sha256)
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3, serialize_spans
from src.training import set_seed, train_epoch, predict, save_checkpoint

import wandb
from time import perf_counter
from src.experiment_tracking import start_run, log_epoch, log_evaluation, finish_run


## W&B login


In [4]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


## Configuration



In [18]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 128, "hidden_size": 256,
    "num_layers": 3, "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 5, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "AdamW",
    "tokenizer": "bpe_bytelevel.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v1", "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "raw", "upload_checkpoint": False,
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_id = "stacked_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]

provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "source_sha256": {path.name: file_sha256(path) for path in sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": file_sha256(repo_dir / "notebooks/04_bilstm_stacked.ipynb"),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
print(device, run_dir)



cuda /content/outputs/stacked_20261112T084900Z_7c1467a9


## Experiment tracking


In [19]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bilstm", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"], repo_dir=repo_dir,
    split_manifest_path=splits_dir / "validation_manifest.parquet",
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", f"layers-{config['num_layers']}", f"hidden-{config['hidden_size']}"]
)
save_json(run_dir / "wandb_run.json", {"id": run.id, "url": run.url,
                                      "path": f"{run.entity}/{run.project}/{run.id}"})
print(run.url)



https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/runs/1ku9diis


## Data



In [7]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir)
display(pd.crosstab(manifest["source"], manifest["split"]))
del manifest

train_records, val_records, alignment = [], [], []
for source in config["sources"]:
    source_splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))
    for split in ("train", "val"):
        frame = source_splits.pop(f"{source}_{split}")[["text", "loc_markers"]]
        records, skipped = prepare_ner_records(tokenizer, frame, source, split,
                                               batch_size=config["prepare_batch_size"])
        (train_records if split == "train" else val_records).extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records, skipped
        gc.collect()
    del source_splits
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)



100%|██████████| 811M/811M [00:47<00:00, 17.9MB/s]

Extracting files...


split,train,val
source,,
ru,100000,49369
uk,805983,201498


Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/101 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/25 [00:00<?, ?it/s]

,source,split,total,retained,alignment_failures
0,uk,train,805983,804381,1602
1,uk,val,201498,201498,361
2,ru,train,100000,99890,110
3,ru,val,49369,49369,44


In [8]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)



In [20]:
"train_records" in globals(), "train_loader" in globals()

(True, True)

## Model



In [21]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"], "num_labels": len(LABEL_TO_ID)
}
model = BiLSTMTagger(**model_config).to(device)
optimizer_class = getattr(torch.optim, config["optimizer"])
optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"])
save_json(run_dir / "model_config.json", model_config)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

run.config.update({"model_config": model_config})


BiLSTMTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
)
Parameters: 8041987


## Training



In [22]:
history, best_score = [], -1
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device)
    scores = competition_scores(val_records, predictions)
    clean_scores = competition_scores(val_records, predictions, cleaner=clean_locations_v3)
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "epoch_seconds": perf_counter() - epoch_started,
                    **{f"{source}_val_loss": value for source, value in val_losses.items()},
                    **scores, **{f"clean_{key}": value for key, value in clean_scores.items()}})
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    save_checkpoint(run_dir / f"epoch_{epoch:02d}.pt", model, optimizer, epoch, config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score = scores[config["selection_metric"]]
        save_checkpoint(best_path, model, optimizer, epoch, config, model_config, scores, provenance)
    log_epoch(run, epoch, history[-1], selection_metric=config["selection_metric"],
              epoch_seconds=history[-1]["epoch_seconds"],
              extra_metrics={f"val_{config['validation_version']}/clean_v3/{key}": value
                             for key, value in clean_scores.items()})
    print(history[-1])
    del predictions

display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()


Train:   0%|          | 0/7110 [00:01<?, ?it/s]

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.034421986565863864, 'epoch_seconds': 179.25916706700036, 'uk_val_loss': 0.019341790940115088, 'ru_val_loss': 0.02527761545431887, 'combined_val_loss': 0.020829338982885425, 'uk_comp_f1': 0.87063619, 'ru_comp_f1': 0.8658051, 'combined_comp_f1': 0.86942891, 'clean_uk_comp_f1': 0.87055453, 'clean_ru_comp_f1': 0.8658051, 'clean_combined_comp_f1': 0.86936634}


Train:   0%|          | 0/7119 [00:01<?, ?it/s]

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.016746668076215228, 'epoch_seconds': 179.34100176399988, 'uk_val_loss': 0.016539924764441535, 'ru_val_loss': 0.019971571099191107, 'combined_val_loss': 0.01739991292219816, 'uk_comp_f1': 0.88908113, 'ru_comp_f1': 0.90750376, 'combined_comp_f1': 0.89372227, 'clean_uk_comp_f1': 0.8888074, 'clean_ru_comp_f1': 0.90752351, 'clean_combined_comp_f1': 0.89352365}


... (7 outputs left out)

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.007547613705417201, 'epoch_seconds': 178.377746956, 'uk_val_loss': 0.018535168596152334, 'ru_val_loss': 0.02195380317191813, 'combined_val_loss': 0.019391895940425848, 'uk_comp_f1': 0.89381552, 'ru_comp_f1': 0.91972103, 'combined_comp_f1': 0.90036122, 'clean_uk_comp_f1': 0.89360389, 'clean_ru_comp_f1': 0.91978098, 'clean_combined_comp_f1': 0.90021852}


,epoch,train_loss,epoch_seconds,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1,clean_uk_comp_f1,clean_ru_comp_f1,clean_combined_comp_f1
0,1,0.034422,179.259167,0.019342,0.025278,0.020829,0.870636,0.865805,0.869429,0.870555,0.865805,0.869366
1,2,0.016747,179.341002,0.016540,0.019972,0.017400,0.889081,0.907504,0.893722,0.888807,0.907524,0.893524
2,3,0.012476,179.770558,0.016390,0.019310,0.017122,0.886359,0.912511,0.892948,0.886047,0.912574,0.892731
3,4,0.009584,178.145993,0.017964,0.021488,0.018847,0.893446,0.913025,0.898396,0.893198,0.913044,0.898216
4,5,0.007548,178.377747,0.018535,0.021954,0.019392,0.893816,0.919721,0.900361,0.893604,0.919781,0.900219


18

## Select checkpoint


In [23]:
checkpoint_path = run_dir / "epoch_03.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model = BiLSTMTagger(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Selected epoch:", checkpoint["epoch"])
summary = {"selected_epoch": checkpoint["epoch"], "checkpoint": checkpoint_path.name,
           "selection_scores": checkpoint["scores"]}
del checkpoint

predictions, val_losses = predict(model, val_loader, device)
summary.update(raw=competition_scores(val_records, predictions),
               clean_v3=competition_scores(val_records, predictions, cleaner=clean_locations_v3),
               validation_losses=val_losses)
save_json(run_dir / "summary.json", summary)
save_predictions(run_dir / "val_predictions_selected.jsonl", val_records, predictions)
print(summary)
del predictions

# log_evaluation(run, dataset=f"val_{config['validation_version']}", metrics=summary["raw"],
#                postprocessing_version="raw", kind="evaluation")
# log_evaluation(run, dataset=f"val_{config['validation_version']}", metrics=summary["clean_v3"],
#                postprocessing_version="clean_v3", kind="evaluation")



Selected epoch: 3


Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

{'selected_epoch': 3, 'checkpoint': 'epoch_03.pt', 'selection_scores': {'uk_comp_f1': 0.88635934, 'ru_comp_f1': 0.91251079, 'combined_comp_f1': 0.89294758}, 'raw': {'uk_comp_f1': 0.88635934, 'ru_comp_f1': 0.91251079, 'combined_comp_f1': 0.89294758}, 'clean_v3': {'uk_comp_f1': 0.88604707, 'ru_comp_f1': 0.91257361, 'combined_comp_f1': 0.89273097}, 'validation_losses': {'uk': 0.01639003412651938, 'ru': 0.019310363276896357, 'combined': 0.017121883577508608}}


## Diagnostics

Test-like examples are validation-derived. Labeling sample is used only for diagnostics.



In [24]:
diagnostic_frames = {
    name: load_manifest(splits_dir, name=name)
    for name in ("uk_testlike_original", "uk_testlike")
}
diagnostic_frames["labeling_sample"] = pd.read_csv(data_dir / "labeling_sample.csv")
diagnostic_scores = []
for name, frame in diagnostic_frames.items():
    frame = frame.copy()
    frame["loc_markers"] = frame["loc_markers"].map(serialize_spans)
    if "row_id" in frame.columns:
        frame = frame.set_index("row_id")
    source = "uk" if name.startswith("uk_") else "labeling"
    records, skipped = prepare_ner_records(tokenizer, frame, source, name)
    predictions, _ = predict(model, make_loader(records, **loader_options), device)
    diagnostic_scores.append({
        "dataset": name, "rows": len(records), "alignment_failures": len(skipped),
        "raw_f1": competition_scores(records, predictions)["combined_comp_f1"],
        "clean_v3_f1": competition_scores(records, predictions, cleaner=clean_locations_v3)["combined_comp_f1"]
    })
    save_predictions(run_dir / f"{name}_predictions.jsonl", records, predictions)
    # log_evaluation(run, dataset=name, metrics={"comp_f1": diagnostic_scores[-1]["raw_f1"]},
    #                postprocessing_version="raw")
    # log_evaluation(run, dataset=name, metrics={"comp_f1": diagnostic_scores[-1]["clean_v3_f1"]},
    #                postprocessing_version="clean_v3")
    del records, predictions
pd.DataFrame(diagnostic_scores).to_csv(run_dir / "diagnostics.csv", index=False)
display(pd.DataFrame(diagnostic_scores))



Prepare uk/uk_testlike_original:   0%|          | 0/3 [00:00<?, ?it/s]

Predict:   0%|          | 0/40 [00:00<?, ?it/s]

Prepare uk/uk_testlike:   0%|          | 0/1 [00:00<?, ?it/s]

Predict:   0%|          | 0/9 [00:00<?, ?it/s]

Prepare labeling/labeling_sample:   0%|          | 0/1 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

,dataset,rows,alignment_failures,raw_f1,clean_v3_f1
0,uk_testlike_original,5000,10,0.884856,0.886641
1,uk_testlike,1000,10,0.784356,0.794069
2,labeling_sample,26,1,0.484211,0.574713


## Test inference



In [25]:
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_predictions, _ = predict(model, make_loader(test_records, **loader_options), device)
save_predictions(run_dir / "test_predictions.jsonl", test_records, test_predictions)

for filename, cleaner in [("submission.csv", list), ("submission_clean_v3.csv", clean_locations_v3)]:
    submission = test[["text_id"]].copy()
    submission["locations"] = submission["text_id"].map(
        lambda text_id: json.dumps(cleaner(test_predictions[text_id]["locations"]), ensure_ascii=False))
    submission.to_csv(run_dir / filename, index=False)
    print(run_dir / filename)
display(submission.head())



Predict:   0%|          | 0/6 [00:00<?, ?it/s]

/content/outputs/stacked_20261112T084900Z_7c1467a9/submission.csv
/content/outputs/stacked_20261112T084900Z_7c1467a9/submission_clean_v3.csv


,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""Договір..."
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова, 9""]"


## Post-processing

In [30]:
!git -C "{repo_dir}" pull --ff-only origin stacked

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 676 bytes | 676.00 KiB/s, done.
From https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning
 * branch            stacked    -> FETCH_HEAD
   4e157cd..f423a80  stacked    -> origin/stacked
Updating 4e157cd..f423a80
Fast-forward
 src/postprocessing.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [31]:
import importlib
import src.postprocessing

importlib.reload(src.postprocessing)
from src.postprocessing import postprocess

In [32]:
from IPython.display import display, Markdown
from src.postprocessing import postprocess

locations, changes = [], []

for row in test.itertuples(index=False):
    spans = test_predictions[row.text_id]["spans"]
    before = postprocess(row.text, spans)
    after = postprocess(
        row.text, spans,
        expand=True, trim=True, empty=True,
    )
    locations.append(after)

    if before != after:
        changes.append({
            "text_id": row.text_id,
            "text": row.text,
            "before": before,
            "after": after,
        })

for change in changes:
    display(Markdown(f"**text_id: {change['text_id']}**"))
    print(change["text"])
    print("BEFORE:", json.dumps(change["before"], ensure_ascii=False))
    print("AFTER: ", json.dumps(change["after"], ensure_ascii=False))
    print()

print("Changed messages:", len(changes))

submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(values, ensure_ascii=False) for values in locations
]

path = run_dir / "submission_clean_v4_combined.csv"
submission.to_csv(path, index=False)

pd.DataFrame(changes).to_json(
    run_dir / "postprocessing_changes.jsonl",
    orient="records", lines=True, force_ascii=False,
)
print(path)

**text_id: 21**

Відео та фото з місця падіння уламків у Дарницькому районі столиці

Попередньо без постраждалих. Уламки спричинили пожежу на даху ТЦ, рятувальники швидко приборкали вогонь. 

«Уламки у Дарницькому районі влучили у будівлю торговельного центру Ашан. Негайно на місце події прибула чергова бригада вогнеборців і ліквідувала наслідки пожежі. Постраждалі особи попередньо відсутні. Зруйновано дах, рештки ракети у внутрішньому приміщенні, працює слідчо-оперативна група», - написав тимчасовий виконувач обов’язків голови Дарницької райдержадміністрації Микола Калашник.

🇺🇦 Украина Сейчас / Подписаться
BEFORE: ["Дарницькому районі столиці", "Ц", "«Уламки", "Дарницькому районі", "Под"]
AFTER:  ["Дарницькому районі столиці", "Уламки", "Дарницькому районі", "Под"]



**text_id: 30**

☀️А ви знали, що біля Києво-Могилянської академії є сонячний годинник? 
Прийти та подивитися як він працює на нього краще у сонячну погоду.  Цей незвичайний пам’ятник науки кінця XVIII століття розташований у внутрішньому дворі колишнього Братського Богоявленського монастиря.
BEFORE: ["Києво", "Братського Богоявленського монастиря"]
AFTER:  ["Києво-Могилянської", "Братського Богоявленського монастиря"]



**text_id: 39**

На Київщині та околицях столиці буде чутно вибухи

У Білоцерківському районі сьогодні впродовж дня проводитимуть знищення вибухонебезпечних предметів.

Так, може бути гучно, але вибухи планові, тому не хвилюйтесь.
ㅤ
  Хуйовий Київ | Надіслати новину
BEFORE: ["Київщині", "У", "Білоцерківському районі"]
AFTER:  ["Київщині", "Білоцерківському районі"]



... (88 outputs left out)

**text_id: 463**

Посеред білого дня крали квіти з міського квітника😠

У Солом’янському районі поліціянти помітили двох чоловіків і жінку, які копирсалися на клумбі і нахабно викопували квіти. Крадіїв затримали і викликали на місце слідчих. 

Патрульні нагадують, що крадіжка, скоєна в умовах воєнного стану, карається позбавленням волі на строк від 5 до 8 років. А за знищення або пошкодження зелених насаджень або інших об’єктів озеленення, що не віднесені до лісового фонду — штраф від 170 до 510 гривень.
BEFORE: ["Солом’янському районі поліці"]
AFTER:  ["Солом’янському районі поліціянти"]



**text_id: 465**

⚠️ Мережею ширяться дописи про зниклих українських військових, чиї тіла «привезли на Київщину». Пояснюємо, що з ними не так

У соцмережі Facebook біля 21 тисяч поширень отримав допис користувачки Amira Tereschuk, яка стверджує, що 274 тіла українських військових доставили в Київську область у червні та липні. Буцімто їх особи не можуть встановити, тому родичів зниклих військових просять повідомляти про їх особливі прикмети.

Пости з абсолютно ідентичними текстами почали розповсюджувати у соцмережах і з інших сторінок Facebook.

Перш за все, викликає питання сторінка, на якій опублікували оголошення, що отримало найбільшу кількість поширень. Вона належить викладачці  східних танців, а не особі, яка займається поверненням чи опізнанням тіл загиблих захисників. На жодному офіційному сайті держслуб немає подібної інформації.

Ба більше, цей же допис «гуляв» соцмережею Facebook рівно рік тому – в серпні 2022-го. Наразі ця інформація неактуальна, а номер телефону, згаданий у дописі, – поза з

## Test chunking

In [15]:
# import re

# chunk_chars = 200
# test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
# chunks = []

# for row in test.itertuples(index=False):
#     for line in re.finditer(r"[^\r\n]+", row.text):
#         start, end = line.span()
#         while start < end:
#             stop = min(start + chunk_chars, end)
#             if stop < end:
#                 space = row.text.rfind(" ", start, stop)
#                 if space > start:
#                     stop = space
#             text = row.text[start:stop]
#             if text.strip():
#                 chunks.append({
#                     "text_id": len(chunks), "text": text,
#                     "original_id": row.text_id, "start": start,
#                 })
#             start = stop

# chunks = pd.DataFrame(chunks)
# chunk_records = prepare_inference_records(tokenizer, chunks)
# chunk_predictions, _ = predict(
#     model, make_loader(chunk_records, **loader_options), device
# )

In [16]:
# chunked_predictions = {
#     text_id: {"spans": [], "locations": []}
#     for text_id in test["text_id"]
# }

# for row in chunks.itertuples(index=False):
#     chunked_predictions[row.original_id]["spans"].extend(
#         (a + row.start, b + row.start)
#         for a, b in chunk_predictions[row.text_id]["spans"]
#     )

# for row in test.itertuples(index=False):
#     prediction = chunked_predictions[row.text_id]
#     prediction["spans"] = sorted(set(prediction["spans"]))
#     prediction["locations"] = [
#         row.text[a:b] for a, b in prediction["spans"]
#     ]

# chunk_dir = run_dir / f"chunked_{checkpoint_path.stem}_{chunk_chars}"
# chunk_dir.mkdir(exist_ok=True)
# save_json(chunk_dir / "config.json", {
#     "checkpoint": str(checkpoint_path), "chunk_chars": chunk_chars,
#     "split": "lines_then_spaces",
# })

# for filename, cleaner in [
#     ("submission_chunked.csv", list),
#     ("submission_chunked_clean_v3.csv", clean_locations_v3),
# ]:
#     submission = test[["text_id"]].copy()
#     submission["locations"] = submission["text_id"].map(
#         lambda text_id: json.dumps(
#             cleaner(chunked_predictions[text_id]["locations"]),
#             ensure_ascii=False,
#         )
#     )
#     submission.to_csv(chunk_dir / filename, index=False)
#     print(chunk_dir / filename)

# display(submission.head())

## Export



In [17]:
finish_run(run, best_epoch=max(history, key=lambda row: row[config["selection_metric"]])["epoch"],
           best_score=best_score,
           selection_metric=config["selection_metric"], checkpoint_path=best_path,
           upload_checkpoint=config["upload_checkpoint"])


import shutil

archive = shutil.make_archive(str(root / run_id), "zip", root_dir=run_dir)
print(archive)
# Colab: download the archive from the Files panel before ending the session.
# Kaggle: save a version with outputs and download the archive from the Output panel.



epoch,▁▃▅▆█
time/epoch_seconds,▆▁▄▄█
train/loss,█▃▂▂▁
val_v1/clean_v3/combined_comp_f1,▁▆▇██
val_v1/clean_v3/ru_comp_f1,▁▆▇▇█
val_v1/clean_v3/uk_comp_f1,▁▆▆██
val_v1/combined_comp_f1,▁▆▇██
val_v1/combined_loss,█▁▁▄▃
val_v1/ru_comp_f1,▁▆▇▇█
val_v1/ru_loss,█▁▁▃▂
+2,...


/content/stacked_20261112T081418Z_c8c1dee5.zip
